# 05 — ETL del corpus español

Extracción del corpus de tweets en español sobre fraude financiero, para el análisis de transferibilidad del enfoque a España.

## Diseño (paralelo al corpus EEUU para comparabilidad)

- **Idioma:** `lang:es`.
- **Geografía:** SIN filtro geográfico nativo. Se filtra España después en pandas vía `user_location` (misma estrategia que el corpus EEUU del notebook 01).
- **Ventana temporal:** todo 2025 (idéntica a EEUU).
- **5 categorías** equivalentes a EEUU: phishing, payment_apps, crypto, romance, impersonation.

## Términos adaptados al contexto español

- Apps de pago → **Bizum** (equivalente español a Zelle/Venmo).
- Organismos → **Hacienda, Correos, Seguridad Social, DGT**.
- Incluye coloquiales (timo, me la han colado) y estafas típicas españolas
  ("hijo en apuros", paquetes en aduanas, falso técnico de la luz).

## Coste

- Hasta 40 llamadas API (5 queries x 8 páginas).
- ⚠️ Volumen esperado MENOR que EEUU (menos población hispanohablante en X).


In [ ]:
# Imports y token
import os
import time
import requests
import pandas as pd
import re
from dotenv import load_dotenv

pd.set_option('display.max_colwidth', None)

load_dotenv()
TOKEN = os.getenv("X_BEARER_TOKEN", "")
if not TOKEN:
    raise ValueError("Falta X_BEARER_TOKEN en .env")

BASE_URL = "https://api.x.com/2/tweets/search/all"


In [ ]:
def search_all_posts_to_df(
    bearer_token, query, max_pages=8, max_results=500,
    sleep_seconds=1.1, start_time=None, end_time=None, label=""
):
    headers = {"Authorization": f"Bearer {bearer_token}"}
    params = {
        "query": query,
        "max_results": max_results,
        "tweet.fields": "id,text,created_at,author_id,lang,geo,public_metrics,entities",
        "expansions": "author_id,geo.place_id",
        "user.fields": "id,name,username,created_at,location,verified,public_metrics",
        "place.fields": "id,full_name,country,country_code,place_type,geo",
        "sort_order": "recency",
    }
    if start_time: params["start_time"] = start_time
    if end_time: params["end_time"] = end_time

    all_rows = []
    next_token = None

    for page in range(max_pages):
        if next_token:
            params["next_token"] = next_token
        else:
            params.pop("next_token", None)

        retries = 0
        while True:
            r = requests.get(BASE_URL, headers=headers, params=params, timeout=30)
            if r.status_code == 200:
                break
            elif r.status_code == 429:
                wait = min(60 * (2 ** retries), 900)
                print(f"  Rate limit. Esperando {wait}s...")
                time.sleep(wait)
                retries += 1
                if retries > 5: raise RuntimeError("Demasiados reintentos")
            elif r.status_code == 400:
                raise RuntimeError(f"400 Bad Request. Query len={len(query)}. {r.text}")
            elif r.status_code == 403:
                raise RuntimeError(f"403 Forbidden. {r.text}")
            else:
                raise RuntimeError(f"X API error {r.status_code}: {r.text}")

        payload = r.json()
        data = payload.get("data", []) or []
        includes = payload.get("includes", {}) or {}
        meta = payload.get("meta", {}) or {}
        users_by_id = {u["id"]: u for u in includes.get("users", []) or []}
        places_by_id = {p["id"]: p for p in includes.get("places", []) or []}

        for t in data:
            author = users_by_id.get(t.get("author_id"), {})
            place_id = (t.get("geo") or {}).get("place_id")
            place = places_by_id.get(place_id, {}) if place_id else {}
            entities = t.get("entities") or {}
            metrics = t.get("public_metrics") or {}
            am = author.get("public_metrics") or {}

            all_rows.append({
                "tweet_id": t.get("id"),
                "created_at": t.get("created_at"),
                "text": t.get("text"),
                "lang": t.get("lang"),
                "author_id": t.get("author_id"),
                "username": author.get("username"),
                "name": author.get("name"),
                "user_location": author.get("location"),
                "user_verified": author.get("verified"),
                "user_followers": am.get("followers_count"),
                "user_tweet_count": am.get("tweet_count"),
                "place_id": place_id,
                "place_full_name": place.get("full_name"),
                "place_country": place.get("country"),
                "place_country_code": place.get("country_code"),
                "place_type": place.get("place_type"),
                "retweet_count": metrics.get("retweet_count"),
                "reply_count": metrics.get("reply_count"),
                "like_count": metrics.get("like_count"),
                "quote_count": metrics.get("quote_count"),
                "n_hashtags": len(entities.get("hashtags", []) or []),
                "n_mentions": len(entities.get("mentions", []) or []),
                "n_urls": len(entities.get("urls", []) or []),
                "query_label": label,
            })

        print(f"  [{label} | p{page+1}/{max_pages}] {len(all_rows)} acumulados")
        next_token = meta.get("next_token")
        if not next_token:
            print(f"  [{label}] sin más páginas")
            break
        time.sleep(sleep_seconds)

    return pd.DataFrame(all_rows)


## Queries en español

5 categorías paralelas a EEUU. Sin filtro geográfico nativo (se filtra España después).


In [ ]:
# Exclusiones mínimas (política, deportes que generan ruido)
EX_MIN = "-elecciones -psoe -pp -vox -sumar -futbol -laliga"
GEO_OPS = "lang:es -is:retweet -is:reply -is:nullcast"

def build(positive):
    q = f"({positive}) {EX_MIN} {GEO_OPS}"
    if len(q) > 512:
        raise ValueError(f"Query excede 512 chars: {len(q)} -> {q}")
    return q

QUERIES = {
    # 1. Phishing / smishing / vishing  (incluye "hijo en apuros")
    "phishing": build(
        '"correo de phishing" OR "SMS fraudulento" OR "mensaje fraudulento" '
        'OR "smishing" OR "vishing" OR "SMS falso" OR "enlace fraudulento" '
        'OR "correo falso" OR "hijo en apuros" OR "he cambiado de numero" '
        'OR "estafa por WhatsApp"'
    ),
    # 2. Apps de pago - Bizum
    "payment_apps": build(
        '"estafa Bizum" OR "fraude Bizum" OR "timo Bizum" '
        'OR "me han estafado con Bizum" OR "estafa PayPal" '
        'OR "me la han colado con Bizum" OR "engano Bizum"'
    ),
    # 3. Cripto / inversión
    "crypto": build(
        '"estafa cripto" OR "estafa criptomonedas" OR "estafa inversion" '
        'OR "chiringuito financiero" OR "rug pull" OR "estafa bitcoin" '
        'OR "timo cripto" OR "estafa trading"'
    ),
    # 4. Estafa amorosa
    "romance": build(
        '("estafa amorosa" OR "estafa sentimental" OR "estafa romantica" '
        'OR "timo del amor") '
        '(dinero OR envie OR transferi OR transferencia OR mande)'
    ),
    # 5. Suplantación de organismos + aduanas + falso técnico
    "impersonation": build(
        '"estafa Hacienda" OR "SMS Agencia Tributaria" OR "estafa Correos" '
        'OR "SMS Correos" OR "estafa Seguridad Social" OR "estafa DGT" '
        'OR "paquete retenido" OR "paquete en aduanas" OR "estafa SEUR" '
        'OR "falso tecnico" OR "estafa soporte tecnico" OR "timo Correos"'
    ),
}

print("=== QUERIES EN ESPAÑOL ===\n")
for label, q in QUERIES.items():
    print(f"{label:<16} {len(q):>4} chars")
    print(f"  {q}\n")


## Confirmación de la tirada

⚠️ Escribe **LANZAR** en el cuadro de texto que aparece arriba del notebook.


In [ ]:
PAGES_PER_QUERY = 8
START = "2025-01-01T00:00:00Z"
END   = "2025-12-31T23:59:59Z"

est_calls = len(QUERIES) * PAGES_PER_QUERY
print("=" * 60)
print("   TIRADA CORPUS ESPAÑOL")
print("=" * 60)
print(f"  Páginas por query:   {PAGES_PER_QUERY}")
print(f"  Categorías:          {len(QUERIES)}")
print(f"  Llamadas API:        hasta {est_calls}")
print(f"  Ventana:             {START} -> {END}")
print(f"  Idioma:              español (lang:es)")
print("=" * 60)
print()
print("⚠️ Filtrado de España se hará DESPUÉS en pandas (user_location).")
print()
confirmation = input("Escribe LANZAR para ejecutar, cualquier otra cosa para abortar: ")
if confirmation.strip().upper() != "LANZAR":
    raise RuntimeError("Tirada abortada por el usuario.")
print("\n✓ Confirmación recibida. Iniciando...\n")


## Ejecución

In [ ]:
dfs = []
for label, query in QUERIES.items():
    print(f"\n>>> {label}")
    df_part = search_all_posts_to_df(
        bearer_token=TOKEN, query=query,
        max_pages=PAGES_PER_QUERY, max_results=500,
        sleep_seconds=1.1, start_time=START, end_time=END,
        label=label,
    )
    print(f">>> {label}: {len(df_part)}")
    dfs.append(df_part)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\n=== TOTAL BRUTO ESPAÑOL: {len(df_raw)} ===")
print("\nPor categoría:")
print(df_raw["query_label"].value_counts())


## Guardado inmediato del bruto

In [ ]:
os.makedirs("../data/raw", exist_ok=True)
df_raw.to_csv("../data/raw/scam_es_FINAL_raw.csv", index=False, encoding="utf-8")
print(f"✓ Guardado: ../data/raw/scam_es_FINAL_raw.csv ({len(df_raw)} filas)")
print()
print("🚨 HAZ COPIA DE SEGURIDAD en Drive antes de seguir.")


## Deduplicación

In [ ]:
df_raw["query_labels"] = df_raw.groupby("tweet_id")["query_label"].transform(
    lambda s: ",".join(sorted(set(s)))
)
df_dedup = df_raw.drop_duplicates(subset="tweet_id", keep="first").reset_index(drop=True)
df_dedup = df_dedup.drop(columns=["query_label"])
print(f"Únicos tras deduplicar: {len(df_dedup)}")


## Filtro geográfico: España

Detectamos España vía `user_location` (ciudades, provincias, gentilicios).


In [ ]:
es_keywords = [
    "españa", "spain", "espana",
    # ciudades grandes
    "madrid", "barcelona", "valencia", "sevilla", "zaragoza", "malaga",
    "murcia", "palma", "bilbao", "alicante", "cordoba", "valladolid",
    "vigo", "gijon", "granada", "coruña", "coruna", "vitoria", "santander",
    "pamplona", "almeria", "san sebastian", "donostia", "burgos", "albacete",
    "salamanca", "huelva", "logroño", "logrono", "badajoz", "leon",
    "tarragona", "cadiz", "lleida", "girona", "toledo", "caceres",
    # comunidades / regiones
    "andalucia", "cataluña", "cataluna", "catalunya", "galicia",
    "euskadi", "pais vasco", "asturias", "cantabria", "aragon",
    "castilla", "extremadura", "navarra", "rioja", "canarias",
    "baleares", "murcia", "comunidad valenciana",
    # gentilicios / otros
    "madrileño", "catalan", "andaluz", "gallego", "vasco",
    "🇪🇸",
]

def looks_spain(row):
    loc = " " + str(row.get("user_location") or "").lower() + " "
    return any(k in loc for k in es_keywords)

df_dedup["likely_spain"] = df_dedup.apply(looks_spain, axis=1)
print(f"Tweets probablemente de España: {df_dedup['likely_spain'].sum()} / {len(df_dedup)}")
print(f"  ({df_dedup['likely_spain'].mean()*100:.1f}%)")


## Filtros estructurales

Mismos filtros que el corpus EEUU, adaptados al español.


In [ ]:
STRONG_FRAUD_TERMS = re.compile(
    r"\b(estafa|estafad|timo|timad|fraude|fraudulent|phishing|smishing|"
    r"vishing|suplant|enga\u00f1|hackeo|hackead|ponzi|piramidal|"
    r"chiringuito|scam)\b",
    re.IGNORECASE,
)
MONEY_TERMS = re.compile(
    r"(\b(dinero|euros?|pasta|pago|pagu|cobr|transferenc|transfer|"
    r"banco|cuenta|tarjeta|bizum|paypal|cripto|bitcoin|reembolso|"
    r"devoluci\u00f3n|estaf|perd|v\u00edctima)\b|\u20ac\s*\d+|\d+\s*euros?)",
    re.IGNORECASE,
)

URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")
def clean_for_length(text):
    if not isinstance(text, str): return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

# Política española (para descartar ruido político)
SOFT_POLITICS = re.compile(
    r"\b(sanchez|feijoo|abascal|ayuso|psoe|pp\b|vox|podemos|sumar|"
    r"gobierno|moncloa|congreso|diputad|ministr)\b",
    re.IGNORECASE,
)

df = df_dedup.copy()
df["clean_text"]   = df["text"].apply(clean_for_length)
df["clean_len"]    = df["clean_text"].str.len()
df["n_words"]      = df["clean_text"].str.split().str.len().fillna(0).astype(int)
df["hashtag_ratio"] = (df["n_hashtags"] / df["n_words"].replace(0,1)).fillna(0)
df["has_strong_fraud"] = df["text"].fillna("").apply(lambda t: bool(STRONG_FRAUD_TERMS.search(t)))
df["has_money"]        = df["text"].fillna("").apply(lambda t: bool(MONEY_TERMS.search(t)))
df["is_soft_politics"] = df["text"].fillna("").apply(lambda t: bool(SOFT_POLITICS.search(t)))
df["semantically_relevant"] = df["has_strong_fraud"] | df["has_money"]

mask = (
    (df["clean_len"] >= 40) &
    (df["semantically_relevant"]) &
    (df["likely_spain"]) &
    (df["hashtag_ratio"] < 0.3) &
    (~df["is_soft_politics"])
)
df_clean = df[mask].reset_index(drop=True)

print("=" * 50)
print("   RESUMEN CORPUS ESPAÑOL")
print("=" * 50)
print(f"  Brutos:              {len(df_raw)}")
print(f"  Únicos:              {len(df_dedup)}")
print(f"  Probablemente España:{df_dedup['likely_spain'].sum()}")
print(f"  Limpios finales:     {len(df_clean)}")
print("=" * 50)
print("\nPor categoría:")
for label in QUERIES.keys():
    n = df_clean["query_labels"].fillna("").str.contains(label).sum()
    print(f"  {label:<16} {n:>5}")


## Guardado del corpus español limpio

In [ ]:
df_clean.to_csv("../data/raw/scam_es_FINAL_clean.csv", index=False, encoding="utf-8")
print(f"✓ Guardado: ../data/raw/scam_es_FINAL_clean.csv ({len(df_clean)} filas)")
print()
print("📌 SIGUIENTE PASO: clasificación con mDeBERTa multilingüe.")


## Inspección visual

In [ ]:
n_muestra = min(20, len(df_clean))
if n_muestra > 0:
    print(f"=== MUESTRA DE {n_muestra} TWEETS ESPAÑOLES LIMPIOS ===\n")
    for _, row in df_clean.sample(n_muestra, random_state=42).iterrows():
        print(f"[{row['query_labels']}] @{row['username']} — {row['user_location']}")
        print(f"  {str(row['text'])[:280]}")
        print()
else:
    print("⚠️ El corpus limpio está vacío. Revisar filtros o volumen de la tirada.")
